# 06 — Analysis: compute metrics, render figures, cross-model comparison

For each model run we have, compute every metric and save under
`results/metrics/<model>/`. Then load Jeff's published Qwen-Omni-7B
metrics from his repo for a side-by-side comparison.

**Runtime:** seconds.
**GPU:** not needed.


In [ ]:
# ─── Colab bootstrap ────────────────────────────────────────
# Mount Drive (caches model weights + videos across sessions),
# clone the repo, install dependencies, and configure the cache dirs.
import os, sys, subprocess, pathlib

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/avut"
    REPO_DIR = "/content/omnimodel-research"
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    if not os.path.exists(REPO_DIR):
        subprocess.run([
            "git", "clone",
            "https://github.com/samadasyed/omnimodel-research.git",
            REPO_DIR,
        ], check=True)
else:
    DRIVE_ROOT = os.path.expanduser("~/avut")
    REPO_DIR = str(pathlib.Path.cwd())
    os.makedirs(DRIVE_ROOT, exist_ok=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Persistent storage layout (data + model caches under DRIVE_ROOT)
DATA_DIR        = os.path.join(DRIVE_ROOT, "data")
VIDEO_DIR       = os.path.join(DATA_DIR, "videos")
AUDIO_DIR       = os.path.join(DATA_DIR, "audio")
SILENT_DIR      = os.path.join(DATA_DIR, "silent")
TRANSCRIPT_DIR  = os.path.join(DATA_DIR, "transcripts")
ANNOTATION_DIR  = os.path.join(DATA_DIR, "annotations")
RESULTS_DIR     = os.path.join(DRIVE_ROOT, "results")
RAW_PRED_DIR    = os.path.join(RESULTS_DIR, "raw_predictions")
METRICS_DIR     = os.path.join(RESULTS_DIR, "metrics")
HF_CACHE        = os.path.join(DRIVE_ROOT, ".cache", "hf")
WHISPER_CACHE   = os.path.join(DRIVE_ROOT, ".cache", "whisper")

for d in [VIDEO_DIR, AUDIO_DIR, SILENT_DIR, TRANSCRIPT_DIR,
          ANNOTATION_DIR, RAW_PRED_DIR, METRICS_DIR, HF_CACHE, WHISPER_CACHE]:
    os.makedirs(d, exist_ok=True)

# Redirect HF cache to Drive so we don't redownload weights each session
os.environ["HF_HOME"] = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE
os.environ["HF_DATASETS_CACHE"] = HF_CACHE

print(f"Repo:       {REPO_DIR}")
print(f"Drive root: {DRIVE_ROOT}")
print(f"HF cache:   {HF_CACHE}")


In [ ]:
%pip install -q matplotlib pandas scipy

In [ ]:
import json, glob
from src import metrics

# Discover model runs we have predictions for
model_runs = sorted(glob.glob(os.path.join(RAW_PRED_DIR, "*")))
model_runs = [m for m in model_runs if os.path.isdir(m)]
print("Found prediction sets:")
for m in model_runs:
    print(f"  {m}")


In [ ]:
def load_stages(pred_dir):
    stages = {}
    for path in sorted(glob.glob(os.path.join(pred_dir, "*.json"))):
        name = os.path.splitext(os.path.basename(path))[0]
        with open(path) as f:
            stages[name] = json.load(f)
    return stages


all_metrics = {}
for run_dir in model_runs:
    name = os.path.basename(run_dir)
    print(f"\n=== {name} ===")
    stages = load_stages(run_dir)
    for s, rows in stages.items():
        valid = sum(1 for r in rows if r.get("predicted_answer") is not None)
        print(f"  {s:30s} {len(rows):4d} rows  ({valid} parseable)")
    report = metrics.compute_all(stages)
    all_metrics[name] = report

    out_dir = os.path.join(METRICS_DIR, name)
    os.makedirs(out_dir, exist_ok=True)
    for fname, data in report.items():
        with open(os.path.join(out_dir, f"{fname}.json"), "w") as f:
            json.dump(data, f, indent=2)
    print(f"  → metrics saved to {out_dir}")


## Headline summary

In [ ]:
def show_overall(report, name):
    print(f"\n=== {name} — overall ===")
    acc = report["accuracy_per_task_per_stage"]
    print("Accuracy by stage:")
    for stage in sorted(acc.keys()):
        ov = acc[stage].get("OVERALL", {})
        if ov.get("accuracy") is not None:
            print(f"  {stage:30s} {ov[\'accuracy\']:.3f}")
    afs_o = report["attribution_faithfulness"].get("OVERALL", {})
    if afs_o:
        print(f"\nAFS overall: {afs_o.get(\'afs\')}  "
              f"(F={afs_o.get(\'faithful\')}, C={afs_o.get(\'confabulated\')}, "
              f"T={afs_o.get(\'trivial\')}, U={afs_o.get(\'unparseable\')})")
    lor_o = report["lexical_override_rate"].get("OVERALL", {})
    if lor_o:
        print(f"LOR overall: {lor_o.get(\'lor\')}  "
              f"(flipped={lor_o.get(\'flipped\')}, stayed={lor_o.get(\'stayed\')})")
    tib_overall = report["transcript_injection_bias"].get("OVERALL", {})
    if tib_overall:
        print(f"TIB overall: {tib_overall.get(\'tib\')}")


for name, report in all_metrics.items():
    show_overall(report, name)


## Cross-model comparison: ours vs Jeff's published Qwen-Omni

Loads Jeff's metrics JSONs from his GitHub for a side-by-side.

In [ ]:
import urllib.request

# Pull Jeff's published metrics directly from GitHub
JEFF_BASE = "https://raw.githubusercontent.com/jjwang8/639_avut/main/results/metrics"

def fetch_jeff(name):
    try:
        with urllib.request.urlopen(f"{JEFF_BASE}/{name}.json", timeout=30) as r:
            return json.loads(r.read())
    except Exception as e:
        print(f"  ✗ {name}: {e}")
        return None

jeff_acc = fetch_jeff("accuracy_per_task_per_stage")
jeff_afs = fetch_jeff("attribution_faithfulness")
jeff_tib = fetch_jeff("transcript_injection_bias")
jeff_drops = fetch_jeff("confidence_drops")

print("Loaded Jeff's metrics" if jeff_acc else "(skipping comparison)")


In [ ]:
import pandas as pd

def cross_model_accuracy_table(all_metrics, jeff_acc):
    rows = []
    if jeff_acc:
        for stage, per_task in jeff_acc.items():
            ov = per_task.get("OVERALL", {})
            rows.append({"model": "qwen2.5-omni-7b (jeff)", "stage": stage,
                         "accuracy": ov.get("accuracy"),
                         "n_valid": ov.get("n_valid")})
    for model_name, report in all_metrics.items():
        for stage, per_task in report["accuracy_per_task_per_stage"].items():
            ov = per_task.get("OVERALL", {})
            rows.append({"model": model_name, "stage": stage,
                         "accuracy": ov.get("accuracy"),
                         "n_valid": ov.get("n_valid")})
    return pd.DataFrame(rows).pivot(index="stage", columns="model",
                                     values="accuracy")


tbl = cross_model_accuracy_table(all_metrics, jeff_acc)
print("Overall accuracy by stage (cross-model):")
print(tbl.round(3).to_string())


In [ ]:
def cross_model_afs_table(all_metrics, jeff_afs):
    rows = []
    if jeff_afs:
        for task, d in jeff_afs.items():
            rows.append({"model": "qwen2.5-omni-7b (jeff)", "task": task,
                         "afs": d.get("afs"), "n_falsifiable": d.get("n_falsifiable")})
    for model_name, report in all_metrics.items():
        for task, d in report["attribution_faithfulness"].items():
            rows.append({"model": model_name, "task": task,
                         "afs": d.get("afs"), "n_falsifiable": d.get("n_falsifiable")})
    return pd.DataFrame(rows).pivot(index="task", columns="model", values="afs")


afs_tbl = cross_model_afs_table(all_metrics, jeff_afs)
print("AFS by task (cross-model):")
print(afs_tbl.round(3).to_string())


## Figures — accuracy heatmap, AFS by task, ΔConf by task

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from src.data_utils import TASK_NAME_TO_CODE

FIG_DIR = os.path.join(RESULTS_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

def plot_accuracy_heatmap(report, name):
    acc = report["accuracy_per_task_per_stage"]
    stages_order = [s for s in [
        "S1_text_only","S2_audio_only","S3_visual_only","S4_full_av",
        "S5_transcript_injected","S6_attribution","S7_mismatched_transcript",
        "S8_prosody"] if s in acc]
    tasks_order = list(TASK_NAME_TO_CODE.keys())
    mat = np.full((len(tasks_order), len(stages_order)), np.nan)
    for j, stage in enumerate(stages_order):
        for i, task in enumerate(tasks_order):
            v = acc.get(stage, {}).get(task, {}).get("accuracy")
            if v is not None:
                mat[i, j] = v
    fig, ax = plt.subplots(figsize=(10, 4))
    im = ax.imshow(mat, vmin=0.0, vmax=1.0, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(stages_order)))
    ax.set_xticklabels(stages_order, rotation=30, ha="right")
    ax.set_yticks(range(len(tasks_order)))
    ax.set_yticklabels([TASK_NAME_TO_CODE[t] for t in tasks_order])
    for i in range(len(tasks_order)):
        for j in range(len(stages_order)):
            if not np.isnan(mat[i, j]):
                ax.text(j, i, f"{mat[i, j]:.2f}", ha="center", va="center",
                        color="white" if mat[i, j] < 0.5 else "black",
                        fontsize=8)
    plt.colorbar(im, ax=ax, label="accuracy")
    ax.set_title(f"Accuracy heatmap — {name}")
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, f"accuracy_heatmap_{name}.png"), dpi=140)
    plt.show()


for name, report in all_metrics.items():
    plot_accuracy_heatmap(report, name)


Figures saved to `RESULTS_DIR/figures/`. Use them in the paper.